# Essentiality: comparison with Evo (genomic language model)

Scores each E. coli K-12 (taxid 83333) and S. cerevisiae (taxid 580240) gene with Evo
(`evo-1-131k-base`): the log-likelihood ratio of a genomic window centred on the gene with
vs without stop codons inserted after the start codon. Needs the `evo-model` package (own
environment), network access (NCBI Entrez, UniProt) and, for yeast, the SGD S288C R64-5-1
genome and GFF (see the experiment README). Outputs are printed, not saved.


In [ ]:
import os
from pathlib import Path

# Same --data-dir as the rest of the experiment (all_fasta_noduplicates2/, label_to_ess7/)
DATA_DIR = Path(os.environ.get("PROTEOMELM_DATA_ROOT", "data")) / "essentiality"
# SGD S288C_reference_genome_R64-5-1_20240529 (unpacked), used by the yeast cell
COMPARISON_DIR = Path("../data/comparison_data")
DEVICE = "cuda:0"
# NCBI asks for a contact address with every Entrez request
ENTREZ_EMAIL = "your_email@example.com"

In [ ]:
from evo import Evo
from evo.scoring import score_sequences
import torch

device = DEVICE

evo_model = Evo('evo-1-131k-base')
model, tokenizer = evo_model.model, evo_model.tokenizer
model.to(device)
model.eval()

##############################

sequence = 'ACGT'
input_ids = torch.tensor(
    tokenizer.tokenize(sequence),
    dtype=torch.int,
).to(device).unsqueeze(0)

with torch.no_grad():
    logits, _ = model(input_ids) # (batch, length, vocab)

# print('Logits: ', logits)
# print('Shape (batch, length, vocab): ', logits.shape)

##############################

sequences = ["ACGT", "AGTA"]
print(score_sequences(seqs=sequences,
                      model=model,
                      tokenizer=tokenizer,
                      device=device))

In [ ]:
import requests
import time
from Bio import Entrez, SeqIO
import json
import pickle

# Set your email for NCBI (required)
Entrez.email = ENTREZ_EMAIL

WINDOW_SIZE = 65536

def get_gene_location_from_uniprot(uniprot_id):
    """Get genomic location info from UniProt ID"""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    
    response = requests.get(url)
    if response.status_code != 200:
        return None
    
    data = response.json()
    
    # Extract gene name and genomic location
    gene_name = None
    if 'genes' in data and len(data['genes']) > 0:
        gene_name = data['genes'][0].get('geneName', {}).get('value')
    
    # Get genomic coordinates from cross-references
    genomic_info = None
    if 'uniProtKBCrossReferences' in data:
        for xref in data['uniProtKBCrossReferences']:
            if xref['database'] == 'RefSeq':
                for prop in xref.get('properties', []):
                    if prop['key'] == 'NucleotideSequenceId':
                        genomic_info = {
                            'refseq_id': prop['value'],
                            'gene_name': gene_name
                        }
                        break
    
    return genomic_info

def get_gene_coordinates(refseq_id, gene_name):
    """Get exact gene coordinates from NCBI"""
    try:
        # Search for the gene in the nucleotide database
        handle = Entrez.esearch(db="nucleotide", term=refseq_id)
        record = Entrez.read(handle)
        handle.close()
        
        if not record['IdList']:
            return None
        
        # Fetch the full GenBank record
        handle = Entrez.efetch(db="nucleotide", id=record['IdList'][0], 
                              rettype="gb", retmode="text")
        gb_record = SeqIO.read(handle, "genbank")
        handle.close()
        
        # Find the gene feature
        for feature in gb_record.features:
            if feature.type == "CDS":
                qualifiers = feature.qualifiers
                if gene_name and 'gene' in qualifiers:
                    if qualifiers['gene'][0] == gene_name:
                        start = int(feature.location.start)
                        end = int(feature.location.end)
                        strand = feature.location.strand
                        
                        return {
                            'refseq_id': refseq_id,
                            'ncbi_id': record['IdList'][0],
                            'start': start,
                            'end': end,
                            'strand': strand,
                            'gene_length': end - start,
                            'genome_length': len(gb_record.seq)
                        }
        
        return None
        
    except Exception as e:
        print(f"Error fetching coordinates: {e}")
        return None

def get_centered_sequence(ncbi_id, start, end, genome_length):
    """Extract a 65,536 bp window centered on the gene"""
    gene_center = (start + end) // 2
    half_window = WINDOW_SIZE // 2
    
    # Calculate window boundaries
    window_start = max(0, gene_center - half_window)
    window_end = min(genome_length, gene_center + half_window)
    
    # Adjust if we hit genome boundaries
    if window_start == 0:
        window_end = min(WINDOW_SIZE, genome_length)
    elif window_end == genome_length:
        window_start = max(0, genome_length - WINDOW_SIZE)
    
    try:
        # Fetch the sequence window
        handle = Entrez.efetch(db="nucleotide", id=ncbi_id,
                              rettype="fasta", retmode="text",
                              seq_start=window_start + 1,  # NCBI uses 1-based
                              seq_stop=window_end)
        
        record = SeqIO.read(handle, "fasta")
        handle.close()
        
        return {
            'sequence': str(record.seq),
            'window_start': window_start,
            'window_end': window_end,
            'gene_position_in_window': start - window_start,
            'actual_window_size': window_end - window_start
        }
        
    except Exception as e:
        print(f"Error fetching sequence: {e}")
        return None

def process_uniprot_id(uniprot_id):
    """Complete pipeline for one UniProt ID"""
    
    # Step 1: Get genomic info from UniProt
    genomic_info = get_gene_location_from_uniprot(uniprot_id)
    if not genomic_info:
        print(f"  Could not find genomic info for {uniprot_id}")
        return None
    
    # print(f"  Gene: {genomic_info['gene_name']}, RefSeq: {genomic_info['refseq_id']}")
    
    # Step 2: Get exact coordinates
    time.sleep(0.4)  # Be nice to NCBI servers
    coords = get_gene_coordinates(genomic_info['refseq_id'], 
                                  genomic_info['gene_name'])
    if not coords:
        print(f"  Could not find coordinates")
        return None
    
    # print(f"  Coordinates: {coords['start']}-{coords['end']} ({coords['gene_length']} bp)")
    
    # Step 3: Get centered sequence
    time.sleep(0.4)
    seq_data = get_centered_sequence(coords['ncbi_id'], coords['start'], 
                                     coords['end'], coords['genome_length'])
    if not seq_data:
        print(f"  Could not fetch sequence")
        return None
    
    # print(f"  Window: {seq_data['window_start']}-{seq_data['window_end']} ({seq_data['actual_window_size']} bp)")
    
    return {
        'uniprot_id': uniprot_id,
        'gene_name': genomic_info['gene_name'],
        'refseq_id': genomic_info['refseq_id'],
        'coordinates': coords,
        'sequence_data': seq_data
    }

# Example usage
#if __name__ == "__main__":
# Example E. coli UniProt IDs
taxid = 83333
fasta_path = f"{DATA_DIR}/all_fasta_noduplicates2/uniprotkb_data_taxid{taxid}.fasta"
label_path = f"{DATA_DIR}/label_to_ess7/labeled_essentiality_taxid{taxid}.pkl"

with open(label_path, "rb") as f:
    ess_labels = pickle.load(f)

results = []
for record in SeqIO.parse(fasta_path, format="fasta"):
    uniprot_id = record.id.split("|")[1]
    print(f"Processing {uniprot_id}... (essentiality = {ess_labels[record.id]['Essentiality']})")

    #test_ids = ["P77434", "P77467"]  # Add your UniProt IDs here

    result = process_uniprot_id(uniprot_id)
    if result:
        results.append(result)
    
print(results)

# # Save results
# with open('ecoli_sequences.json', 'w') as f:
#     json.dump(results, f, indent=2)

# print(f"Processed {len(results)} proteins successfully")

In [ ]:
import requests
from Bio import Entrez, SeqIO
import json
import gzip
from io import StringIO, BytesIO

# Set your email for NCBI (required)
Entrez.email = ENTREZ_EMAIL

WINDOW_SIZE = 65536
ECOLI_REFSEQ = "NC_000913.3"  # E. coli K-12 MG1655

def download_ecoli_genome():
    """Download the complete E. coli genome GenBank file with all features"""
    print(f"Downloading {ECOLI_REFSEQ} from NCBI FTP...")
    
    # Method 1: Direct download from NCBI FTP (most reliable)
    ftp_url = "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gbff.gz"
    
    try:
        print("Downloading from FTP site (this may take a minute)...")
        response = requests.get(ftp_url, stream=True)
        
        if response.status_code == 200:
            # Decompress and parse
            with gzip.open(BytesIO(response.content), 'rt') as f:
                record = SeqIO.read(f, "genbank")
            
            print(f"Downloaded genome: {len(record.seq):,} bp")
            print(f"Description: {record.description}")
            print(f"Number of features: {len(record.features):,}")
            
            # Count CDS features
            cds_count = sum(1 for f in record.features if f.type == "CDS")
            print(f"CDS features found: {cds_count:,}")
            
            if cds_count == 0:
                print("WARNING: No CDS features found!")
                return None
            
            return record
        else:
            print(f"FTP download failed with status {response.status_code}")
            
    except Exception as e:
        print(f"FTP download error: {e}")
    
    # Method 2: Try Entrez as fallback
    print("\nTrying Entrez API as fallback...")
    try:
        handle = Entrez.efetch(db="nucleotide", id=ECOLI_REFSEQ,
                              rettype="gbwithparts", retmode="text")
        record = SeqIO.read(handle, "genbank")
        handle.close()
        
        print(f"Downloaded genome: {len(record.seq):,} bp")
        print(f"Number of features: {len(record.features):,}")
        
        cds_count = sum(1 for f in record.features if f.type == "CDS")
        print(f"CDS features found: {cds_count:,}")
        
        if cds_count > 0:
            return record
        else:
            print("WARNING: No CDS features in Entrez download")
            
    except Exception as e:
        print(f"Entrez error: {e}")
    
    print("\nERROR: Could not download genome with features.")
    print("Please download manually from:")
    print(ftp_url)
    print("Then use: record = SeqIO.read('GCF_000005845.2_ASM584v2_genomic.gbff', 'genbank')")
    return None

def extract_cds_features(genome_record):
    """Extract all CDS features with Swiss-Prot IDs and coordinates"""
    print("\nExtracting CDS features...")
    
    cds_list = []
    no_swissprot = 0
    
    for feature in genome_record.features:
        if feature.type == "CDS":
            qualifiers = feature.qualifiers
            
            # Get Swiss-Prot ID
            swissprot_id = None
            if 'db_xref' in qualifiers:
                for xref in qualifiers['db_xref']:
                    if xref.startswith('UniProtKB/Swiss-Prot:'):
                        swissprot_id = xref.split(':')[1]
                        break
            
            if not swissprot_id:
                no_swissprot += 1
                continue
            
            # Extract coordinates
            start = int(feature.location.start)
            end = int(feature.location.end)
            strand = feature.location.strand  # 1 for forward, -1 for reverse
            
            # Extract other useful information
            gene_name = qualifiers.get('gene', [None])[0]
            locus_tag = qualifiers.get('locus_tag', [None])[0]
            product = qualifiers.get('product', [None])[0]
            protein_id = qualifiers.get('protein_id', [None])[0]
            
            cds_list.append({
                'uniprot_id': swissprot_id,
                'gene_name': gene_name,
                'locus_tag': locus_tag,
                'product': product,
                'protein_id': protein_id,
                'start': start,
                'end': end,
                'strand': strand,
                'length': end - start
            })
    
    print(f"Found {len(cds_list)} CDS features with Swiss-Prot IDs")
    print(f"Skipped {no_swissprot} CDS features without Swiss-Prot IDs")
    
    return cds_list

def get_centered_sequence(genome_seq, start, end, strand):
    """Extract a 65,536 bp window centered on the gene"""
    genome_length = len(genome_seq)
    gene_center = (start + end) // 2
    half_window = WINDOW_SIZE // 2
    
    # Calculate window boundaries
    window_start = max(0, gene_center - half_window)
    window_end = min(genome_length, gene_center + half_window)
    
    # Adjust if we hit genome boundaries
    actual_window_size = window_end - window_start
    if actual_window_size < WINDOW_SIZE:
        if window_start == 0:
            window_end = min(WINDOW_SIZE, genome_length)
        elif window_end == genome_length:
            window_start = max(0, genome_length - WINDOW_SIZE)
        actual_window_size = window_end - window_start
    
    # Extract sequence
    sequence = str(genome_seq[window_start:window_end])
    
    # Calculate gene position within window
    gene_start_in_window = start - window_start
    gene_end_in_window = end - window_start
    
    return {
        'sequence': sequence,
        'window_start': window_start,
        'window_end': window_end,
        'gene_start_in_window': gene_start_in_window,
        'gene_end_in_window': gene_end_in_window,
        'actual_window_size': actual_window_size,
        'strand': strand
    }


def process_all_proteins(genome_record, save_sequences=True):
    """Process all proteins and extract their centered sequences"""
    
    # Extract all CDS features
    cds_list = extract_cds_features(genome_record)
    
    print(f"\nExtracting {WINDOW_SIZE} bp windows for all genes...")
    
    results = {}
    for i, cds in enumerate(cds_list):
        
        # Get centered sequence
        seq_data = get_centered_sequence(
            genome_record.seq,
            cds['start'],
            cds['end'],
            cds['strand']
        )
        
        result = {
            'uniprot_id': cds['uniprot_id'],
            'gene_name': cds['gene_name'],
            'locus_tag': cds['locus_tag'],
            'product': cds['product'],
            'protein_id': cds['protein_id'],
            'genomic_start': cds['start'],
            'genomic_end': cds['end'],
            'strand': cds['strand'],
            'gene_length': cds['length'],
            'window_start': seq_data['window_start'],
            'window_end': seq_data['window_end'],
            'gene_start_in_window': seq_data['gene_start_in_window'],
            'gene_end_in_window': seq_data['gene_end_in_window'],
            'actual_window_size': seq_data['actual_window_size']
        }
        
        if save_sequences:
            result['sequence'] = seq_data['sequence']
        
        results[cds['uniprot_id']] = result
    
    print(f"Completed processing {len(results)} genes")
    return results

def save_results(results, save_full_sequences=True):
    """Save results to files"""
    
    # Save metadata (without full sequences for smaller file)
    print("\nSaving metadata...")
    metadata = []
    for r in results:
        meta = {k: v for k, v in r.items() if k != 'sequence'}
        if 'sequence' in r:
            meta['sequence_length'] = len(r['sequence'])
        metadata.append(meta)
    
    with open('ecoli_cds_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved metadata to ecoli_cds_metadata.json")
    
    # Save full sequences in FASTA format
    if save_full_sequences and 'sequence' in results[0]:
        print("Saving sequences in FASTA format...")
        with open('ecoli_centered_sequences.fasta', 'w') as f:
            for r in results:
                header = (f">{r['uniprot_id']}|{r['gene_name']}|{r['locus_tag']}|"
                         f"genome:{r['window_start']}-{r['window_end']}|"
                         f"gene_in_window:{r['gene_start_in_window']}-{r['gene_end_in_window']}|"
                         f"strand:{'+' if r['strand'] == 1 else '-'}")
                f.write(f"{header}\n")
                
                # Write sequence in 80-character lines
                seq = r['sequence']
                for i in range(0, len(seq), 80):
                    f.write(f"{seq[i:i+80]}\n")
        
        print(f"Saved sequences to ecoli_centered_sequences.fasta")
    
    # Create a simple mapping file
    print("Saving UniProt ID to coordinates mapping...")
    with open('uniprot_to_coords.tsv', 'w') as f:
        f.write("uniprot_id\tgene_name\tlocus_tag\tstart\tend\tstrand\tgene_length\n")
        for r in results:
            f.write(f"{r['uniprot_id']}\t{r['gene_name']}\t{r['locus_tag']}\t"
                   f"{r['genomic_start']}\t{r['genomic_end']}\t{r['strand']}\t{r['gene_length']}\n")
    print(f"Saved mapping to uniprot_to_coords.tsv")

# Main execution
if __name__ == "__main__":
    taxid = 83333
    fasta_path = f"{DATA_DIR}/all_fasta_noduplicates2/uniprotkb_data_taxid{taxid}.fasta"
    label_path = f"{DATA_DIR}/label_to_ess7/labeled_essentiality_taxid{taxid}.pkl"

    with open(label_path, "rb") as f:
        ess_labels = pickle.load(f)
    
    # Download the E. coli genome
    genome = download_ecoli_genome()
    
    if genome:
        # Process all proteins
        uniprot_to_seq = process_all_proteins(genome, save_sequences=True)

        results = []
        i = 0
        for record in SeqIO.parse(fasta_path, format="fasta"):
            uniprot_id = record.id.split("|")[1]

            print(uniprot_to_seq[uniprot_id])
            i += 1
            if i > 4:
                break

In [ ]:
#% ECOLI

import requests
from Bio import Entrez, SeqIO
import json
import gzip
from io import StringIO, BytesIO
import pickle
from evo import Evo
from evo.scoring import score_sequences
import torch

# Set your email for NCBI (required)
Entrez.email = ENTREZ_EMAIL

WINDOW_SIZE = 8192 #65536 # 8192

def download_ecoli_genome():

    ECOLI_REFSEQ = "NC_000913.3"  # E. coli K-12 MG1655
    
    """Download the complete E. coli genome GenBank file with all features"""
    print(f"Downloading {ECOLI_REFSEQ} from NCBI FTP...")
    
    # Method 1: Direct download from NCBI FTP (most reliable)
    ftp_url = "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gbff.gz"
    
    try:
        print("Downloading from FTP site (this may take a minute)...")
        response = requests.get(ftp_url, stream=True)
        
        if response.status_code == 200:
            # Decompress and parse
            with gzip.open(BytesIO(response.content), 'rt') as f:
                record = SeqIO.read(f, "genbank")
            
            print(f"Downloaded genome: {len(record.seq):,} bp")
            print(f"Description: {record.description}")
            print(f"Number of features: {len(record.features):,}")
            
            # Count CDS features
            cds_count = sum(1 for f in record.features if f.type == "CDS")
            print(f"CDS features found: {cds_count:,}")
            
            if cds_count == 0:
                print("WARNING: No CDS features found!")
                return None
            
            return record
        else:
            print(f"FTP download failed with status {response.status_code}")
            
    except Exception as e:
        print(f"FTP download error: {e}")
    
    # Method 2: Try Entrez as fallback
    print("\nTrying Entrez API as fallback...")
    try:
        handle = Entrez.efetch(db="nucleotide", id=ECOLI_REFSEQ,
                              rettype="gbwithparts", retmode="text")
        record = SeqIO.read(handle, "genbank")
        handle.close()
        
        print(f"Downloaded genome: {len(record.seq):,} bp")
        print(f"Number of features: {len(record.features):,}")
        
        cds_count = sum(1 for f in record.features if f.type == "CDS")
        print(f"CDS features found: {cds_count:,}")
        
        if cds_count > 0:
            return record
        else:
            print("WARNING: No CDS features in Entrez download")
            
    except Exception as e:
        print(f"Entrez error: {e}")
    
    print("\nERROR: Could not download genome with features.")
    print("Please download manually from:")
    print(ftp_url)
    print("Then use: record = SeqIO.read('GCF_000005845.2_ASM584v2_genomic.gbff', 'genbank')")
    return None

def extract_cds_features(genome_record):
    """Extract all CDS features with Swiss-Prot IDs and coordinates"""
    print("\nExtracting CDS features...")
    
    cds_list = []
    no_swissprot = 0
    
    for feature in genome_record.features:
        if feature.type == "CDS":
            qualifiers = feature.qualifiers
            
            # Get Swiss-Prot ID
            swissprot_id = None
            if 'db_xref' in qualifiers:
                for xref in qualifiers['db_xref']:
                    if xref.startswith('UniProtKB/Swiss-Prot:'):
                        swissprot_id = xref.split(':')[1]
                        break
            
            if not swissprot_id:
                no_swissprot += 1
                continue
            
            # Extract coordinates
            start = int(feature.location.start)
            end = int(feature.location.end)
            strand = feature.location.strand  # 1 for forward, -1 for reverse
            
            # Extract other useful information
            gene_name = qualifiers.get('gene', [None])[0]
            locus_tag = qualifiers.get('locus_tag', [None])[0]
            product = qualifiers.get('product', [None])[0]
            protein_id = qualifiers.get('protein_id', [None])[0]
            
            cds_list.append({
                'uniprot_id': swissprot_id,
                'gene_name': gene_name,
                'locus_tag': locus_tag,
                'product': product,
                'protein_id': protein_id,
                'start': start,
                'end': end,
                'strand': strand,
                'length': end - start
            })
    
    print(f"Found {len(cds_list)} CDS features with Swiss-Prot IDs")
    print(f"Skipped {no_swissprot} CDS features without Swiss-Prot IDs")
    
    return cds_list

def get_centered_sequence(genome_seq, start, end, strand, perturbed=False):
    """Extract a 65,536 bp window centered on the gene"""
    genome_length = len(genome_seq)
    gene_center = (start + end) // 2
    half_window = WINDOW_SIZE // 2
    
    # Calculate window boundaries
    window_start = max(0, gene_center - half_window)
    window_end = min(genome_length, gene_center + half_window)
    
    # Adjust if we hit genome boundaries
    actual_window_size = window_end - window_start
    if actual_window_size < WINDOW_SIZE:
        if window_start == 0:
            window_end = min(WINDOW_SIZE, genome_length)
        elif window_end == genome_length:
            window_start = max(0, genome_length - WINDOW_SIZE)
        actual_window_size = window_end - window_start
    
    # Extract sequence
    sequence = str(genome_seq[window_start:window_end])

    # Calculate gene position within window
    gene_start_in_window = start - window_start
    gene_end_in_window = end - window_start
    
    if perturbed==False:
        return {"unperturbed": sequence, 
                "perturbed": None,
                "gene_start_in_window": gene_start_in_window,
                "gene_end_in_window": gene_end_in_window,
                "init_trim": 0}
    

    if gene_start_in_window <= 7:
        initial_trim = gene_start_in_window
        final_trim = WINDOW_SIZE-(15-initial_trim)
    elif WINDOW_SIZE-gene_end_in_window <= 8:
        final_trim = gene_end_in_window
        initial_trim = 15-(WINDOW_SIZE-final_trim)
    else:
        initial_trim=7
        final_trim=WINDOW_SIZE-8
    
    perturbed_seq = sequence[initial_trim:(gene_start_in_window+12)] + "TAATAATAATAGTGA" + sequence[(gene_start_in_window+12):final_trim]

    return {"unperturbed": sequence, 
            "perturbed": perturbed_seq,
            "gene_start_in_window": gene_start_in_window,
            "gene_end_in_window": gene_end_in_window,
            "init_trim": initial_trim}
    
def process_all_proteins(genome_record):
    """Process all proteins and extract their centered sequences"""
    
    # Extract all CDS features
    cds_list = extract_cds_features(genome_record)
    
    print(f"\nExtracting {WINDOW_SIZE} bp windows for all genes...")
    
    results = {}
    for i, cds in enumerate(cds_list):
        
        # Get centered sequence
        seq_data = get_centered_sequence(
            genome_record.seq,
            cds['start'],
            cds['end'],
            cds['strand'],
            perturbed=True
        )

        results[cds['uniprot_id']] = {"unperturbed_seq": seq_data["unperturbed"],
                                      "perturbed_seq": seq_data["perturbed"],
                                      "gene_start": seq_data["gene_start_in_window"],
                                      "gene_end": seq_data["gene_end_in_window"],
                                      "init_trim": seq_data["init_trim"]}
    
    print(f"Completed processing {len(results)} genes")
    return results

# Main execution
if __name__ == "__main__":
    taxid = 83333
    fasta_path = f"{DATA_DIR}/all_fasta_noduplicates2/uniprotkb_data_taxid{taxid}.fasta"
    label_path = f"{DATA_DIR}/label_to_ess7/labeled_essentiality_taxid{taxid}.pkl"

    with open(label_path, "rb") as f:
        ess_labels = pickle.load(f)
    
    # Download the E. coli genome
    genome = download_ecoli_genome()

    device = DEVICE

    evo_model = Evo('evo-1-131k-base')
    model, tokenizer = evo_model.model, evo_model.tokenizer
    model.to(device)
    model.eval()

    if genome:
        # Process all proteins
        uniprot_to_seq = process_all_proteins(genome)

        results = []
        for record in SeqIO.parse(fasta_path, format="fasta"):
            uniprot_id = record.id.split("|")[1]

            ess = []
            for item in ess_labels[record.id]['Essentiality']:
                if ((item == "E") or (item == "NE")):
                    ess.append(item)
            ess = ess.pop() if ess else "UNK"
            ess_translate = {"E": 0, "NE": 1}
            if ess == "UNK":
                continue

            try:
                seq_data = uniprot_to_seq[uniprot_id]
                # print(seq_data["unperturbed_seq"][seq_data["gene_start"]-5:seq_data["gene_start"]+30])
                # print(seq_data["perturbed_seq"][seq_data["gene_start"]-seq_data["init_trim"]-5:seq_data["gene_start"]-seq_data["init_trim"]+30])
            
            except:
                # print(f"NOT FOUND SEQ DATA FOR UNIPROT ID {uniprot_id} <----------------------------")
                seq_data = None
                
            if seq_data is not None:
                scores_unpert = score_sequences(seqs=[seq_data["unperturbed_seq"]],
                          model=model,
                          tokenizer=tokenizer,
                          device=device)
                
                scores_pert = score_sequences(seqs=[seq_data["perturbed_seq"]],
                          model=model,
                          tokenizer=tokenizer,
                          device=device)
                
                results.append({"uniprot_id": uniprot_id,
                                # "unperturbed_seq": seq_data["unperturbed_seq"],
                                # "perturbed_seq": seq_data["perturbed_seq"],
                                "ess": ess_translate[ess],
                                "evo_score": scores_pert[0]/scores_unpert[0]})

                print(f"Uniprot id {uniprot_id} -- {ess} -- {scores_pert}/{scores_unpert}")            
    
    print(results)

In [ ]:
#% YEAST

import re
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import gzip
from collections import defaultdict
import pickle
from evo import Evo
from evo.scoring import score_sequences

# Step 1: Download required files
# You'll need:
# - Genome: http://sgd-archive.yeastgenome.org/sequence/S288C_reference/genome_releases/S288C_reference_genome_Current_Release.tgz
# - GFF: http://sgd-archive.yeastgenome.org/curation/chromosomal_feature/saccharomyces_cerevisiae.gff

def parse_gff_for_genes(gff_file):
    """
    Parse GFF file to get gene coordinates.
    Returns dict: {gene_id: (chromosome, start, end, strand)}
    """
    gene_info = {}
    
    with open(gff_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            parts = line.strip().split('\t')
            if len(parts) < 9:
                continue
            
            feature_type = parts[2]
            if feature_type != 'gene':
                continue
            
            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            strand = parts[6]
            attributes = parts[8]
            
            # Extract gene ID from attributes
            # Format varies, but typically: ID=gene:YAL001C;Name=TFC3;...
            gene_id = None
            for attr in attributes.split(';'):
                if 'ID=' in attr or 'Name=' in attr:
                    match = re.search(r'(Y[A-Z]{2}\d{3}[WC](?:-[A-Z])?)', attr)
                    if match:
                        gene_id = match.group(1)
                        break
            
            if gene_id:
                gene_info[gene_id] = (chrom, start, end, strand)
    
    return gene_info

def load_genome(genome_fasta):
    """
    Load genome sequences into a dictionary.
    Returns dict: {chromosome: SeqRecord}
    """
    genome = {}
    for record in SeqIO.parse(genome_fasta, "fasta"):
        # Chromosome names might be like "chrI", "chrII", etc.
        
        # Extract chromosome
        match = re.search(r'\[chromosome=([^\]]+)\]', record.description)
        if match:
            chromosome = match.group(1)
        
        genome[f"chr{chromosome}"] = record
    return genome

def extract_centered_window(genome, chrom, start, end, strand, window_size=8192):
    """
    Extract a window of specified size centered on the gene.
    """
    gene_center = (start + end) // 2
    window_start = max(0, gene_center - window_size // 2)
    window_end = window_start + window_size
    
    # Get chromosome sequence
    if chrom not in genome:
        return None
    
    chrom_seq = genome[chrom].seq
    
    # Adjust if window exceeds chromosome end
    if window_end > len(chrom_seq):
        window_end = len(chrom_seq)
        window_start = max(0, window_end - window_size)
    
    # Extract sequence
    seq = chrom_seq[window_start:window_end]

    # If gene is on minus strand, reverse complement
    if strand == '-':
        seq = seq.reverse_complement()
    
        # Calculate gene position within window
        gene_start_in_window = window_size - end + window_start
        gene_end_in_window = window_size - start + window_start
    else:
        # Calculate gene position within window
        gene_start_in_window = start - window_start
        gene_end_in_window = end - window_start

    if gene_start_in_window <= 7:
        initial_trim = gene_start_in_window
        final_trim = window_size-(15-initial_trim)
    elif window_size-gene_end_in_window <= 8:
        final_trim = gene_end_in_window
        initial_trim = 15-(window_size-final_trim)
    else:
        initial_trim=7
        final_trim=window_size-8
    
    perturbed_seq = seq[initial_trim:(gene_start_in_window+12)] + "TAATAATAATAGTGA" + seq[(gene_start_in_window+12):final_trim]

    return {"unperturbed": str(seq), 
            "perturbed": str(perturbed_seq),
            "gene_start_in_window": gene_start_in_window,
            "gene_end_in_window": gene_end_in_window,
            "init_trim": initial_trim}
    
    #return seq, window_start, window_end

def process_one_protein_record(record, gene_info, genome, window_size):
    # Extract SGD ID from header (e.g., >YAL001C or similar)
    match = re.search(r'(Y[A-Z]{2}\d{3}[WC](?:-[A-Z])?)', record.description)
    
    if not match:
        return None
    
    gene_id = match.group(1)
    
    if gene_id not in gene_info:
        return None
    
    chrom, start, end, strand = gene_info[gene_id]
    
    result = extract_centered_window(genome, chrom, start, end, strand, window_size)

    return result

# Usage example:
if __name__ == "__main__":
    taxid = 580240
    protein_fasta = f"{DATA_DIR}/all_fasta_noduplicates2/othersource_data_taxid{taxid}.fasta"
    label_path = f"{DATA_DIR}/label_to_ess7/labeled_essentiality_taxid{taxid}.pkl"

    gff_file = f"{COMPARISON_DIR}/S288C_reference_genome_R64-5-1_20240529/saccharomyces_cerevisiae_R64-5-1_20240529.gff"
    genome_fasta = f"{COMPARISON_DIR}/S288C_reference_genome_R64-5-1_20240529/S288C_reference_sequence_R64-5-1_20240529.fsa"  # or individual chromosome files
    
    with open(label_path, "rb") as f:
        ess_labels = pickle.load(f)

    # Load data
    print("Loading gene annotations...")
    gene_info = parse_gff_for_genes(gff_file)
    
    print("Loading genome...")
    genome = load_genome(genome_fasta)

    device = DEVICE

    evo_model = Evo('evo-1-131k-base')
    model, tokenizer = evo_model.model, evo_model.tokenizer
    model.to(device)
    model.eval()
    
    print("Processing proteins...")
    results = []
    not_found = []
    
    for record in SeqIO.parse(protein_fasta, "fasta"):

        ess = []
        for item in ess_labels[record.id]['Essentiality']:
            if ((item == "E") or (item == "NE")):
                ess.append(item)
        ess = ess.pop() if ess else "UNK"
        ess_translate = {"E": 0, "NE": 1}
        if ess == "UNK":
            continue
        
        seq_data = process_one_protein_record(record, gene_info, genome, window_size=8192)
        
        if seq_data is None:
            not_found.append(record.id)
            continue
        
        scores_unpert = score_sequences(seqs=[seq_data["unperturbed"]],
                          model=model,
                          tokenizer=tokenizer,
                          device=device)
                
        scores_pert = score_sequences(seqs=[seq_data["perturbed"]],
                model=model,
                tokenizer=tokenizer,
                device=device)
        
        results.append({"gene": record.id,
                        # "unperturbed_seq": seq_data["unperturbed_seq"],
                        # "perturbed_seq": seq_data["perturbed_seq"],
                        "ess": ess_translate[ess],
                        "evo_score": scores_pert[0]/scores_unpert[0]})

        print(f"Record id {record.id} -- {ess} -- {scores_pert}/{scores_unpert}")            
    
    if not_found:
        print(f"\nWarning: {len(not_found)} genes not found:")
        print(not_found[:10], "..." if len(not_found) > 10 else "")

    print(results)